# ML Evaluation: Keyword-to-Mood Classification

Academic evaluation of the keyword-to-mood classifier used by the movie recommender.
Covers all course-required ML workflow elements:

1. Data loading and train/val/test split with stratification
2. Feature scaling (RobustScaler, fit on train only)
3. 5+ classifier comparison (scaled vs unscaled)
4. Best model selection and test-set evaluation
5. Confusion matrix visualization
6. 10-fold cross-validation
7. KNN hyperparameter tuning (k=1..20)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler

## 1. Load labeled keyword data

The labeled dataset contains ~5000 TMDB keywords manually assigned to one of 7 Ekman moods:
Happy, Interested, Surprised, Sad, Disgusted, Afraid, Angry.
We use only single-label assignments for clean classification.

In [ ]:
DATA_DIR = Path("../../data")
TSV_PATH = DATA_DIR / "input" / "tmdb-keyword-frequencies_labeled_top5000.tsv"
OUTPUT_DIR = DATA_DIR / "output"

labeled = pd.read_csv(TSV_PATH, sep="\t")
single = labeled[labeled["assignment_type"] == "single"]
keywords = single["keyword_name"].tolist()
labels = single["assigned_moods"].tolist()

print(f"Total labeled keywords: {len(labeled)}")
print(f"Single-label keywords: {len(single)}")
print(f"Label distribution:\n{single['assigned_moods'].value_counts()}")

## 2. Generate embeddings

Using Google's EmbeddingGemma-300M to encode keyword text into 256-dim vectors.
These semantic embeddings capture meaning beyond simple TF-IDF.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-300m")
embeddings = model.encode(keywords, show_progress_bar=True, normalize_embeddings=True)
print(f"Embedding shape: {embeddings.shape}")

## 3. Train / Validation / Test split

- 80% train, 10% validation, 10% test
- Stratified to preserve class distribution
- Test set held out until final evaluation

In [ ]:
le = LabelEncoder()
y = le.fit_transform(labels)
label_names = list(le.classes_)

# First split: 90% train+val, 10% test
x_tv, x_test, y_tv, y_test = train_test_split(
    embeddings, y, test_size=0.10, stratify=y, random_state=13,
)
# Second split: 80% train, 10% val (0.125 of 90% = 10% of total)
x_train, x_val, y_train, y_val = train_test_split(
    x_tv, y_tv, test_size=0.125, stratify=y_tv, random_state=13,
)

print(f"Train: {len(x_train)}, Val: {len(x_val)}, Test: {len(x_test)}")
print(f"Classes: {label_names}")

## 4. Feature scaling

RobustScaler fitted on training data only to prevent data leakage.

In [ ]:
scaler = RobustScaler()
x_train_s = scaler.fit_transform(x_train)
x_val_s = scaler.transform(x_val)
x_test_s = scaler.transform(x_test)

## 5. Classifier comparison (scaled + unscaled)

5 classifiers + 2 DummyClassifier baselines, evaluated on both scaled and unscaled data.
Metrics: accuracy, precision, recall, F1 (all macro-averaged).

In [ ]:
from ml.evaluation import evaluate_classifiers

# Scaled
results_scaled = evaluate_classifiers(x_train_s, x_val_s, y_train, y_val, scaled=True)
# Unscaled
results_unscaled = evaluate_classifiers(x_train, x_val, y_train, y_val, scaled=False)

results = pd.concat([results_scaled, results_unscaled]).sort_values("Val F1", ascending=False)
results

## 6. Best model: test-set evaluation

Select the best classifier by validation F1 (scaled, non-dummy).
Evaluate on the held-out test set: classification report + confusion matrix.

In [ ]:
from ml.evaluation import best_model_report, get_classifiers

# Identify best non-dummy scaled classifier
best_row = results_scaled[
    ~results_scaled["Classifier"].str.startswith("Dummy")
].sort_values("Val F1", ascending=False).iloc[0]

best_name = best_row["Classifier"]
print(f"Best classifier: {best_name}")
print(f"Val Accuracy: {best_row['Val Acc']:.1%}")
print(f"Val F1 (macro): {best_row['Val F1']:.4f}")
print(f"Train-Val Gap: {best_row['Train F1'] - best_row['Val F1']:.4f}")

# Retrain best model and evaluate on test set
classifiers = get_classifiers()
best_clf = classifiers[best_name]
best_clf.fit(x_train_s, y_train)

report, cm_fig = best_model_report(best_clf, x_test_s, y_test, label_names)
print(f"\nClassification Report ({best_name}):\n")
print(report)

In [ ]:
# Confusion matrix
cm_fig

## 7. 10-fold cross-validation

KFold with shuffle on the full dataset to estimate generalization performance.

In [ ]:
from ml.evaluation import run_cross_validation

x_all_s = scaler.fit_transform(embeddings)
cv_scores = run_cross_validation(classifiers[best_name], x_all_s, y)

print(f"10-fold CV ({best_name}):")
print(f"Mean accuracy: {cv_scores.mean():.1%}")
print(f"Std: \u00b1 {cv_scores.std():.1%}")
print(f"Per-fold: {[f'{s:.1%}' for s in cv_scores]}")

## 8. KNN hyperparameter tuning (k=1..20)

Vary k and plot train vs validation accuracy to detect overfitting.

In [ ]:
from ml.evaluation import knn_hyperparameter_plot

k_fig = knn_hyperparameter_plot(x_train_s, x_val_s, y_train, y_val)
k_fig